# Athena emulator — Terraform IaC

You keep this Athena footprint in version control: workgroup, database,
named query, prepared statement, and data catalog are declared in HCL and
`terraform` is your interface to the emulator. Each section runs the real
CLI through `init → apply → plan -detailed-exitcode → destroy` — the same
lifecycle the emulator's own terraform parity suite drives — over its
shipped module (`libs/athena-local/tests/terraform/apply/`), and records
a measured row per capability.

Stack (`docker-compose.yml`, network `mlops_net`): `athena` (emulator) ·
`trino` (query engine) · `moto` (S3/Glue) · `jupyterlab` (this notebook).
Per ADR-0002, only Athena traffic goes to the emulator; S3/Glue calls
must still land on moto.

In [1]:
import os

# Compose sets these for in-container runs; the localhost fallbacks let the
# same notebook run against the published ports from the host.
os.environ.setdefault("AWS_ENDPOINT_URL", "http://localhost:5000")
os.environ.setdefault("AWS_ENDPOINT_URL_ATHENA", "http://localhost:5001")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

ATHENA_URL = os.environ["AWS_ENDPOINT_URL_ATHENA"]
EDGE_URL = os.environ["AWS_ENDPOINT_URL"]

print(f"athena → {ATHENA_URL}")
print(f"s3/glue → {EDGE_URL}")

athena → http://localhost:5001
s3/glue → http://localhost:5000


In [2]:
import json
import shutil
import subprocess
import tempfile
import uuid
from pathlib import Path

import boto3
from _evidence import PROJECT_DIR, persist, record, render
from _helpers import (
    client_error_of,
    delete_run_buckets,
    delete_run_catalogs,
    delete_run_databases,
    delete_workgroup_contents,
    run_workgroup_names,
)

athena = boto3.client("athena")
s3 = boto3.client("s3")
glue = boto3.client("glue")

assert athena.meta.endpoint_url.rstrip("/") == ATHENA_URL
assert s3.meta.endpoint_url.rstrip("/") == EDGE_URL
assert glue.meta.endpoint_url.rstrip("/") == EDGE_URL

RUN_SUFFIX = uuid.uuid4().hex[:8]
# The module prefixes every resource tfapply_*; an nb08_ infix keeps this
# run's names sweepable without colliding with the libs suite's pure-hex
# suffixes. The bucket is notebook-side, so it carries the nb08- prefix.
NAME_SUFFIX = f"nb08_{RUN_SUFFIX}"
bucket = f"nb08-{uuid.uuid4().hex[:12]}"
s3.create_bucket(Bucket=bucket)

REPO_ROOT = PROJECT_DIR.parents[1]
MODULE = REPO_ROOT / "libs" / "athena-local" / "tests" / "terraform" / "apply"
assert MODULE.is_dir(), f"terraform module not found at {MODULE}"

# State and .terraform must not land in the repo: the module is copied to a
# throwaway workdir per run. The workdir sits under the project — a disk
# filesystem (/tmp is tmpfs and the provider binary is ~800 MB), on the
# same fs as the plugin cache so hard-links work, and at a path the docker
# daemon sees identically for the in-container fallback below.
for stale in PROJECT_DIR.glob(".nb08-tf-*"):
    shutil.rmtree(stale, ignore_errors=True)
WORKDIR = Path(tempfile.mkdtemp(prefix=".nb08-tf-", dir=PROJECT_DIR))
shutil.copytree(
    MODULE,
    WORKDIR,
    ignore=shutil.ignore_patterns(".plugin-cache", ".terraform", "*.tfstate*"),
    dirs_exist_ok=True,  # mkdtemp already made the empty target
)

# The committed lock file pins aws 6.65.0; this repo-local cache lets init
# resolve it without touching the registry.
PLUGIN_CACHE = (
    REPO_ROOT
    / "libs"
    / "athena-local"
    / "tests"
    / "terraform"
    / ".plugin-cache"
)
PLUGIN_CACHE.mkdir(parents=True, exist_ok=True)

TF_ENV = {
    **os.environ,
    "TF_IN_AUTOMATION": "1",
    "TF_INPUT": "0",
    # No checkpoint telemetry: version/apply must never block on
    # checkpoint.hashicorp.com.
    "CHECKPOINT_DISABLE": "1",
    "TF_PLUGIN_CACHE_DIR": str(PLUGIN_CACHE),
    "TF_VAR_athena_endpoint": ATHENA_URL,
    "TF_VAR_data_plane_endpoint": EDGE_URL,
    "TF_VAR_bucket": bucket,
    "TF_VAR_name_suffix": NAME_SUFFIX,
}
TF_TIMEOUT_S = 600.0
TERRAFORM_IMAGE = "hashicorp/terraform:1.16.3"
# The compose network — a sibling spawned through the socket mount
# resolves the athena/moto service names in TF_VAR_*_endpoint on it.
DOCKER_NETWORK = "mlops_net"


def terraform_binary() -> Path | None:
    """Configured (``ATHENA_TERRAFORM_BINARY``) or PATH-found terraform."""
    configured = os.environ.get("ATHENA_TERRAFORM_BINARY")
    if configured:
        binary = Path(configured)
        assert binary.is_file(), (
            f"ATHENA_TERRAFORM_BINARY={configured} is not a file"
        )
        return binary
    found = shutil.which("terraform")
    return Path(found) if found else None


def docker_terraform(arguments: list[str]) -> list[str]:
    """``docker run`` wrapping for terraform when the host has no binary —
    the fallback a JupyterLab kernel takes, spawning a sibling container
    through the compose socket mount."""
    env = {
        key: value for key, value in TF_ENV.items() if key.startswith("TF_")
    }
    env.update(
        {
            "TF_PLUGIN_CACHE_DIR": "/tf-cache",
            "HOME": "/tmp",
            "CHECKPOINT_DISABLE": "1",
        }
    )
    return [
        "docker",
        "run",
        "--rm",
        "--network",
        DOCKER_NETWORK,
        # Without --user the sibling writes root-owned state into the
        # workdir and the cleanup cell's rmtree can't remove it.
        "--user",
        f"{os.getuid()}:{os.getgid()}",
        "--volume",
        f"{WORKDIR}:/tf",
        "--workdir",
        "/tf",
        "--volume",
        f"{PLUGIN_CACHE}:/tf-cache",
        *(f"--env={key}={value}" for key, value in env.items()),
        TERRAFORM_IMAGE,
        *arguments,
    ]


def run_terraform(arguments: list[str]) -> subprocess.CompletedProcess[str]:
    """One terraform CLI call — the exit code is data (``plan
    -detailed-exitcode`` answers 2 for drift), never a raised error."""
    if TF_BINARY is not None:
        return subprocess.run(
            [str(TF_BINARY), *arguments],
            cwd=WORKDIR,
            env=TF_ENV,
            capture_output=True,
            text=True,
            timeout=TF_TIMEOUT_S,
        )
    return subprocess.run(
        docker_terraform(arguments),
        capture_output=True,
        text=True,
        timeout=TF_TIMEOUT_S,
    )


def tail(text: str, lines: int = 15) -> str:
    """The last ``lines`` of output — keeps committed cells readable."""
    return "\n".join(text.strip().splitlines()[-lines:])


TF_BINARY = terraform_binary()
if TF_BINARY is None:
    assert shutil.which("docker") is not None, (
        "no terraform binary on PATH and no docker CLI for the "
        f"{TERRAFORM_IMAGE} fallback"
    )

version = run_terraform(["version", "-json"])
assert version.returncode == 0, version.stderr
tf_version = json.loads(version.stdout)["terraform_version"]

record(
    "terraform CLI resolved (host binary, else docker via socket)",
    "PASS",
    f"{TF_BINARY or TERRAFORM_IMAGE} v{tf_version}",
)

print(f"terraform → {TF_BINARY or TERRAFORM_IMAGE} (v{tf_version})")
print(f"workdir={WORKDIR} bucket={bucket} suffix={NAME_SUFFIX}")

terraform → /home/gabriel-menezes/.local/bin/terraform (v1.16.3)
workdir=/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/.nb08-tf-mmv1rjf_ bucket=nb08-beb49bccc73e suffix=nb08_792f2e67


## Init

The committed `.terraform.lock.hcl` pins `hashicorp/aws = 6.65.0`;
`TF_PLUGIN_CACHE_DIR` points at the repo's cache, so `init` resolves the
provider without touching the registry.

In [3]:
init = run_terraform(["init", "-input=false", "-no-color"])
assert init.returncode == 0, (
    f"terraform init failed (exit {init.returncode}):\n"
    f"{init.stdout}\n{init.stderr}"
)
assert "6.65.0" in init.stdout, init.stdout
print(tail(init.stdout))

record(
    "terraform init — aws provider 6.65.0 from committed lock",
    "PASS",
    "plugin cache "
    + ("hit" if "shared cache" in init.stdout else "miss")
    + f" ({PLUGIN_CACHE})",
)

Initializing the backend...

Initializing provider plugins...
- Reusing previous version of hashicorp/aws from the dependency lock file
- Using hashicorp/aws v6.65.0 from the shared cache directory

Terraform has been successfully initialized!


Evidence(feature='terraform init — aws provider 6.65.0 from committed lock', status='PASS', detail='plugin cache hit (/home/gabriel-menezes/Documents/repos/mlops/libs/athena-local/tests/terraform/.plugin-cache)', latency_ms=None)

## Apply

`apply` creates all five `aws_athena_*` resources; `output -json` hands
their names to boto3, which confirms each one landed — enforced workgroup,
Glue database, named query, prepared statement, LAMBDA catalog.

In [4]:
apply_result = run_terraform(
    ["apply", "-auto-approve", "-input=false", "-no-color"]
)
assert apply_result.returncode == 0, (
    f"terraform apply failed (exit {apply_result.returncode}):\n"
    f"{apply_result.stdout}\n{apply_result.stderr}"
)
print(tail(apply_result.stdout))

outputs_proc = run_terraform(["output", "-json"])
assert outputs_proc.returncode == 0, outputs_proc.stderr
outputs = {
    name: str(entry["value"])
    for name, entry in json.loads(outputs_proc.stdout).items()
}

workgroup = athena.get_work_group(WorkGroup=outputs["workgroup_name"])[
    "WorkGroup"
]
assert workgroup["State"] == "ENABLED"
assert workgroup["Configuration"]["EnforceWorkGroupConfiguration"] is True
assert (
    workgroup["Configuration"]["ResultConfiguration"]["OutputLocation"]
    == f"s3://{bucket}/workgroup-results/"
)
glue.get_database(Name=outputs["database_name"])
named = athena.get_named_query(NamedQueryId=outputs["named_query_id"])[
    "NamedQuery"
]
assert named["QueryString"] == "SELECT 1"
statement = athena.get_prepared_statement(
    StatementName=outputs["prepared_statement_name"],
    WorkGroup=outputs["workgroup_name"],
)["PreparedStatement"]
assert statement["QueryStatement"] == "SELECT ?"
catalog = athena.get_data_catalog(Name=outputs["data_catalog_name"])[
    "DataCatalog"
]
assert catalog["Type"] == "LAMBDA"

record(
    "terraform apply — five aws_athena_* resources created",
    "PASS",
    f"{outputs['workgroup_name']}, {outputs['database_name']}, "
    f"{outputs['named_query_id'][:8]}…, "
    f"{outputs['prepared_statement_name']}, {outputs['data_catalog_name']}",
)
record(
    "boto3 witnesses — every resource readable on emulator/moto",
    "PASS",
    f"workgroup ENABLED + OutputLocation=s3://{bucket}/workgroup-results/; "
    "glue db + named query 'SELECT 1' + prepared 'SELECT ?' + "
    "LAMBDA catalog",
)
outputs

aws_athena_database.this: Creating...
aws_athena_prepared_statement.this: Creation complete after 0s [id=tfapply_wg_nb08_792f2e67/tfapply_ps_nb08_792f2e67]
aws_athena_database.this: Creation complete after 3s [id=tfapply_db_nb08_792f2e67]
aws_athena_named_query.this: Creating...
aws_athena_named_query.this: Creation complete after 0s [id=4f5cf994-baa4-410a-b6a5-fc0da6741acf]

Apply complete! Resources: 5 added, 0 changed, 0 destroyed.

Outputs:

data_catalog_name = "tfapply_cat_nb08_792f2e67"
database_name = "tfapply_db_nb08_792f2e67"
named_query_id = "4f5cf994-baa4-410a-b6a5-fc0da6741acf"
prepared_statement_name = "tfapply_ps_nb08_792f2e67"
workgroup_name = "tfapply_wg_nb08_792f2e67"


{'data_catalog_name': 'tfapply_cat_nb08_792f2e67',
 'database_name': 'tfapply_db_nb08_792f2e67',
 'named_query_id': '4f5cf994-baa4-410a-b6a5-fc0da6741acf',
 'prepared_statement_name': 'tfapply_ps_nb08_792f2e67',
 'workgroup_name': 'tfapply_wg_nb08_792f2e67'}

## Drift check

`plan -detailed-exitcode` re-reads every resource after apply: exit 2
would mean drift — a field the provider set but the emulator dropped —
so exit 0 proves the emulator echoes state back verbatim.

In [5]:
plan = run_terraform(
    ["plan", "-detailed-exitcode", "-input=false", "-no-color"]
)
assert plan.returncode == 0, (
    f"terraform plan reports drift or errored (exit {plan.returncode}):\n"
    f"{plan.stdout}\n{plan.stderr}"
)
print(tail(plan.stdout))

record(
    "terraform plan -detailed-exitcode → 0 (zero drift after apply)",
    "PASS",
    "re-read of all five resources matches state — the emulator echoed "
    "the provider's fields verbatim",
)

aws_athena_data_catalog.this: Refreshing state... [id=tfapply_cat_nb08_792f2e67]
aws_athena_workgroup.this: Refreshing state... [id=tfapply_wg_nb08_792f2e67]
aws_athena_prepared_statement.this: Refreshing state... [id=tfapply_wg_nb08_792f2e67/tfapply_ps_nb08_792f2e67]
aws_athena_database.this: Refreshing state... [id=tfapply_db_nb08_792f2e67]
aws_athena_named_query.this: Refreshing state... [id=4f5cf994-baa4-410a-b6a5-fc0da6741acf]

No changes. Your infrastructure matches the configuration.

Terraform has compared your real infrastructure against your configuration
and found no differences, so no changes are needed.


Evidence(feature='terraform plan -detailed-exitcode → 0 (zero drift after apply)', status='PASS', detail="re-read of all five resources matches state — the emulator echoed the provider's fields verbatim", latency_ms=None)

## Destroy

`apply -destroy` unwinds the module through the same APIs; boto3 then
confirms the emulator and moto forgot all five resources — the stack ends
as it began.

In [6]:
destroy = run_terraform(
    ["apply", "-destroy", "-auto-approve", "-input=false", "-no-color"]
)
assert destroy.returncode == 0, (
    f"terraform destroy failed (exit {destroy.returncode}):\n"
    f"{destroy.stdout}\n{destroy.stderr}"
)
print(tail(destroy.stdout))

gone = {
    "workgroup": lambda: athena.get_work_group(
        WorkGroup=outputs["workgroup_name"]
    ),
    "database": lambda: glue.get_database(Name=outputs["database_name"]),
    "named_query": lambda: athena.get_named_query(
        NamedQueryId=outputs["named_query_id"]
    ),
    "prepared_statement": lambda: athena.get_prepared_statement(
        StatementName=outputs["prepared_statement_name"],
        WorkGroup=outputs["workgroup_name"],
    ),
    "data_catalog": lambda: athena.get_data_catalog(
        Name=outputs["data_catalog_name"]
    ),
}
errors = {name: client_error_of(call) for name, call in gone.items()}
assert all(error is not None for error in errors.values()), errors

record(
    "terraform destroy — all five resources absent per boto3",
    "PASS",
    "5/5 read-backs raise ClientError: " + ", ".join(sorted(errors)),
)

  - named_query_id          = "4f5cf994-baa4-410a-b6a5-fc0da6741acf" -> null
  - prepared_statement_name = "tfapply_ps_nb08_792f2e67" -> null
  - workgroup_name          = "tfapply_wg_nb08_792f2e67" -> null
aws_athena_data_catalog.this: Destroying... [id=tfapply_cat_nb08_792f2e67]
aws_athena_prepared_statement.this: Destroying... [id=tfapply_wg_nb08_792f2e67/tfapply_ps_nb08_792f2e67]
aws_athena_named_query.this: Destroying... [id=4f5cf994-baa4-410a-b6a5-fc0da6741acf]
aws_athena_named_query.this: Destruction complete after 0s
aws_athena_data_catalog.this: Destruction complete after 0s
aws_athena_prepared_statement.this: Destruction complete after 0s
aws_athena_database.this: Destroying... [id=tfapply_db_nb08_792f2e67]
aws_athena_database.this: Destruction complete after 3s
aws_athena_workgroup.this: Destroying... [id=tfapply_wg_nb08_792f2e67]
aws_athena_workgroup.this: Destruction complete after 1s

Apply complete! Resources: 0 added, 0 changed, 5 destroyed.


Evidence(feature='terraform destroy — all five resources absent per boto3', status='PASS', detail='5/5 read-backs raise ClientError: data_catalog, database, named_query, prepared_statement, workgroup', latency_ms=None)

## Capabilities demonstrated

Every capability above recorded a measured row. The matrix persists to
`parity/08_terraform_iac.md`; `PARITY.md` at the project root is
regenerated from all fragments.

In [7]:
print(render())
persist("08_terraform_iac")

| feature | status | detail | latency_ms |
|---|---|---|---|
| terraform CLI resolved (host binary, else docker via socket) | PASS | /home/gabriel-menezes/.local/bin/terraform v1.16.3 |  |
| terraform init — aws provider 6.65.0 from committed lock | PASS | plugin cache hit (/home/gabriel-menezes/Documents/repos/mlops/libs/athena-local/tests/terraform/.plugin-cache) |  |
| terraform apply — five aws_athena_* resources created | PASS | tfapply_wg_nb08_792f2e67, tfapply_db_nb08_792f2e67, 4f5cf994…, tfapply_ps_nb08_792f2e67, tfapply_cat_nb08_792f2e67 |  |
| boto3 witnesses — every resource readable on emulator/moto | PASS | workgroup ENABLED + OutputLocation=s3://nb08-beb49bccc73e/workgroup-results/; glue db + named query 'SELECT 1' + prepared 'SELECT ?' + LAMBDA catalog |  |
| terraform plan -detailed-exitcode → 0 (zero drift after apply) | PASS | re-read of all five resources matches state — the emulator echoed the provider's fields verbatim |  |
| terraform destroy — all five resources 

PosixPath('/home/gabriel-menezes/Documents/repos/mlops/projects/athena_emulator/parity/08_terraform_iac.md')

## Clean up

Terraform already destroyed its resources; the run's bucket and the
temporary module copy are notebook-side — delete them, and sweep any
`tfapply_*_nb08_*` leftovers a crashed run may have abandoned.

In [8]:
for workgroup_name in run_workgroup_names(athena, "tfapply_wg_nb08_"):
    delete_workgroup_contents(athena, workgroup_name)
    athena.delete_work_group(WorkGroup=workgroup_name)
delete_run_catalogs(athena, "tfapply_cat_nb08_")
delete_run_databases(glue, "tfapply_db_nb08_")
delete_run_buckets(s3, "nb08-")
shutil.rmtree(WORKDIR, ignore_errors=True)
print("cleaned: nb08 bucket + tfapply_*_nb08_* leftovers + workdir removed")

cleaned: nb08 bucket + tfapply_*_nb08_* leftovers + workdir removed
